In [2]:
from pathlib import Path
from YOLOv8BeyondEarth.predict import get_sliced_prediction
from rastertools_BOULDERING import raster, metadata as raster_metadata, crs as raster_crs
from shptools_BOULDERING import shp, geometry as shp_geom, geomorph as shp_geomorph, annotations as shp_anno, metrics as shp_metrics
from sahi import AutoDetectionModel

## Sliced prediction (get_sliced_prediction)

Function to slice a large image and get prediction for each slice + combine all of the predictions back to the full image.

The time to run the script is dependent on the number of predictions over the whole image. This is because the algorithm loops through each prediction in each slice and transform the bool_mask to polygon. 

**Few notes:**

1. YOLOv8 model expects the image to be in BGR with the following shape (height, width, 3). 
2. If you want to detect very small objects, the slice size should be
   decreased, and the inference size should be increased.
   - slice_size = 256
   - inference_size = 1024.
3. If you are in the opposite situation, where you realized that most of the large boulders are missed out. You can
   increase the slice height and width.
   - slice_size = 1024
   - inference_size  = 512, 1024.

You can get the best of both worlds by combining predictions (2) and (3) with Non-Maximum Suppresion (NMS). Obviously the larger the slice size, and the larger the inference size, the more time it takes to run this script. 

Test Time Augmentation could be included too, but it takes lot of time to run it.. 

Note that the bboxes (in absolute coordinates) are calculated from the bounds of the polygons after the
predictions are computed.

**Example:**

Let's have a look at an example. We want to automatically detect boulders on NAC image M139694087LE, which depicts about half of the Censorinus impact crater on the lunar surface. Censorinus is relatively fresh impact crater with a diameter of about 4 km. More than 250,000 boulders are located close to its vicinity (Krishna et al., 2016). I have provided the NAC image on my GoogleDrive so that you do not have to process it. edictions are computed.redictions are computed.n. 

In [3]:
home_p = Path.home() / "Downloads" / "Test_ManualBoulderNetEnvironment"
work_dir= home_p / "tmp" / "YOLOv8BeyondEarth"
raster_dir = (work_dir / "raster")
model_dir = (work_dir / "yolov8_model")

# Let's define the temporary working directories (feel free to change where the raster is saved to)
work_dir.mkdir(parents=True, exist_ok=True)
raster_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True) 

In [4]:
raster_dir

WindowsPath('C:/Users/Brian/Downloads/Test_ManualBoulderNetEnvironment/tmp/YOLOv8BeyondEarth/raster')

In [5]:
url_raster = "https://drive.google.com/uc?id=1o9A0GSHQ0m_XTPAgDRDTD25xlgWutqMW"
url_yolov8_model_weights = "https://drive.google.com/uc?id=1DJ3Ek4NI1uEzlB1pyor-KDRN8_pEorVp"

You need to `pip install gdown` if needed

In [9]:
import gdown

#### Downloading of example image and model weights

In [10]:
gdown.download(url_raster, (raster_dir / "M139694087LE.tif").as_posix(), quiet=True)
gdown.download(url_yolov8_model_weights, (model_dir / "yolov8-m-boulder-detection-tmp.pt").as_posix(), quiet=True)

'C:/Users/Brian/Downloads/Test_ManualBoulderNetEnvironment/tmp/YOLOv8BeyondEarth/yolov8_model/yolov8-m-boulder-detection-tmp.pt'

In [6]:
in_raster = raster_dir / "M139694087LE.tif"
model_weights = model_dir / "yolov8-m-boulder-detection-tmp.pt"

### Boulder detection
#### Loading of the model

In [7]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=model_weights.as_posix(),
    confidence_threshold=0.1, # this parameter will be changed by the function
    device="cuda:0",  # or cpu, please run this code on a computer with a graphical card, otherwise it will take a long time! 
    image_size=1024) # this parameter will be changed by the function

#### Predictions over the whole NAC image

In order to detect boulders of different sizes, we are running the code for 4 different slice sizes (256x256, 512x512, 768x768 and 1024x1024). The sliced images are then upsampled to 1024x1024 (the inference size). We here used a 20% overlap between the sliced images to avoid for edge artifacts. A NMS threshold is set to 0.20 to filter away overlapping bouding boxes/masks (this step is based on the amount of overlap between bounding boxes). Only boulders with more than 6 pixels are kept. The `downscale_pred` flag controls if the polygons are derived from the masks generated during the inference (1024x1024 in this case), or if they are computed from the resized masks (resized from inference to slice size). Using this flag speed up significantly the code, and it is almost necessary for the code to run at an OK speed for images with lot of predictions. However, the boulder outlines may suffer from this approximation. You can play a bit around with this flag, and see how it impacts the performances.

In [8]:
output_dir = (work_dir / "inference_fast3")
output_dir.mkdir(parents=True, exist_ok=True)

interim_dir = (work_dir / "interim_dir")
interim_dir.mkdir(parents=True, exist_ok=True) 

In [ ]:
import cv2
import geopandas as gpd
import numpy as np
import pandas as pd
import torch

from YOLOv8BeyondEarth.polygon import (binary_mask_to_polygon, is_within_slice, shift_polygon,
                                       add_geometries, bboxes_to_shp, outlines_to_shp)
from lsnms import nms, wbc

from sahi.slicing import slice_image
from tqdm import tqdm
from pathlib import Path

from rastertools_BOULDERING import raster, convert as raster_convert, metadata as raster_metadata
from shptools_BOULDERING import shp


def binary_mask_to_polygon_cv(binary_mask):
    """
    Faster alternative to skimage.measure.find_contours.
    Converts a binary mask (2D array of 0/1) into a polygon contour.

    Args:
        binary_mask (np.ndarray): 2D numpy array, dtype=bool or uint8

    Returns:
        np.ndarray: (N, 2) array of polygon coordinates (x, y)
                    or None if no valid contour is found
    """
    # Ensure correct dtype
    mask_uint8 = (binary_mask.astype(np.uint8) * 255)

    # Find external contours only (no holes)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    # Keep the largest contour (most masks only have one)
    contour = max(contours, key=cv2.contourArea)

    # Remove redundant singleton dimension (OpenCV returns Nx1x2)
    return contour.squeeze(1)



# --- Updated YOLOv8: now processes one prediction result (no model call) ---
def YOLOv8fast(prediction_result, in_slice_info, detection_model, has_mask,
           shift_amount, slice_size, min_area_threshold, downscale_pred):
    """
    Processes a single prediction_result (one element from detection_model.model(batch))
    and returns a DataFrame of detections (same format as before).

    - prediction_result: single result object returned by detection_model.model for one image
    - in_slice_info: (optional) any info about the slice (kept for parity; not used here)
    """
    shift_x = shift_amount[0]
    shift_y = shift_amount[1]

    # prediction_result is a single result (for one image)
    # if no predictions
    if prediction_result.boxes.data.size()[0] == 0:
        df = pd.DataFrame(columns=['score', 'polygon', 'category_id', 'category_name', 'is_within_slice'])
        return df

    # At this point there are predictions
    if has_mask:
        result_boxes = prediction_result.boxes.data[prediction_result.boxes.data[:, 4] >= detection_model.confidence_threshold]
        result_masks = prediction_result.masks.data[prediction_result.boxes.data[:, 4] >= detection_model.confidence_threshold]
    else:
        result_boxes = prediction_result.boxes.data[prediction_result.boxes.data[:, 4] >= detection_model.confidence_threshold]
        result_masks = torch.tensor([[] for _ in range(result_boxes.size()[0])])

    # Prepare containers (same as original)
    scores = []
    polygons = []
    category_ids = []
    category_names = []
    is_polygon_within_slice_list = []

    image_predictions_in_xyxy_format = result_boxes
    image_predictions_masks = result_masks

    # iterate predictions for this single image
    for prediction, bool_mask in zip(
            image_predictions_in_xyxy_format.cpu().detach().numpy(),
            image_predictions_masks.cpu().detach().numpy()
    ):

        score = prediction[4]
        category_id = int(prediction[5])
        category_name = detection_model.category_mapping[str(category_id)]

        # threshold masks (more accurate before resizing)
        if bool_mask.size != 0:
            bool_mask[bool_mask >= 0.5] = 1
            bool_mask[bool_mask < 0.5] = 0

        if downscale_pred:
            if bool_mask.shape[0] == slice_size:
                pass
            else:
                # if bool_mask is empty (no mask), keep as is
                if bool_mask.size != 0:
                    bool_mask = cv2.resize(bool_mask, (slice_size, slice_size), interpolation=cv2.INTER_AREA)

        # number of pixels (guard for empty mask)
        area = 0
        if bool_mask.size != 0:
            area = int(np.count_nonzero(bool_mask == 1))

        if area > min_area_threshold:
            try:
                polygon = binary_mask_to_polygon(bool_mask)
                if downscale_pred:
                    polygon_slice = polygon
                else:
                    polygon_slice = np.stack([(polygon[:, 0] / bool_mask.shape[0]) * slice_size,
                                              (polygon[:, 1] / bool_mask.shape[0]) * slice_size], axis=-1)

                min_edge_distance = 0.05 * slice_size
                max_edge_distance = 0.95 * slice_size
                is_polygon_within_slice = (np.logical_and(polygon_slice[:, 0].min() >= min_edge_distance,
                                                          polygon_slice[:, 0].max() <= max_edge_distance) and
                                           np.logical_and(polygon_slice[:, 1].min() >= min_edge_distance,
                                                          polygon_slice[:, 1].max() <= max_edge_distance))

                if not is_polygon_within_slice:
                    score = 0.10  # if at edge set score to a low value

                # conversion to absolute coordinates,
                shifted_polygon = shift_polygon(polygon_slice, shift_x, shift_y)

                scores.append(score)
                polygons.append(shifted_polygon)  # conversion of polygon to absolute coordinates
                category_ids.append(category_id)
                category_names.append(category_name)
                is_polygon_within_slice_list.append(is_polygon_within_slice)
            except Exception:
                # preserve original behavior of ignoring failures
                pass

    dict_out = {'score': scores, 'polygon': polygons,
                'category_id': category_ids, 'category_name': category_names,
                'is_within_slice': is_polygon_within_slice_list}

    df = pd.DataFrame(dict_out)
    return df

def YOLOv8fastv2(prediction_result, in_slice_info, detection_model, has_mask,
           shift_amount, slice_size, min_area_threshold, downscale_pred):
    shift_x, shift_y = shift_amount
    boxes = prediction_result.boxes.data  # (N,6)
    if boxes.size(0) == 0:
        return pd.DataFrame(columns=['score', 'polygon', 'category_id', 'category_name', 'is_within_slice'])

    # filter on GPU
    conf_mask = boxes[:, 4] >= detection_model.confidence_threshold
    boxes = boxes[conf_mask]
    if has_mask:
        masks = prediction_result.masks.data[conf_mask]
    else:
        masks = torch.empty((boxes.shape[0], 0), device=boxes.device)

    scores, polygons, category_ids, category_names, is_within_slice_list = [], [], [], [], []

    for i in range(len(boxes)):
        score = float(boxes[i, 4])
        category_id = int(boxes[i, 5])
        category_name = detection_model.category_mapping[str(category_id)]

        if has_mask and masks[i].numel() > 0:
            bool_mask = masks[i]
            # threshold mask on GPU
            bool_mask = (bool_mask >= 0.5).float()

            # downscale if needed
            if downscale_pred and bool_mask.shape[-1] != slice_size:
                bool_mask_np = cv2.resize(
                    bool_mask.cpu().numpy(),
                    (slice_size, slice_size),
                    interpolation=cv2.INTER_AREA
                )
                bool_mask = torch.from_numpy(bool_mask_np).to(bool_mask.device)
        else:
            bool_mask = torch.zeros((slice_size, slice_size), device=boxes.device)

        # count area on GPU
        area = int(torch.count_nonzero(bool_mask).item())

        if area <= min_area_threshold:
            continue

        # move mask to CPU only once here
        bool_mask_np = bool_mask.cpu().numpy()
        bool_mask_np = (bool_mask_np > 0).astype(np.uint8)


        try:
            polygon = binary_mask_to_polygon_cv(bool_mask_np)
            if polygon is None:
                continue

            if downscale_pred:
                polygon_slice = polygon
            else:
                polygon_slice = np.stack([
                    (polygon[:, 0] / bool_mask_np.shape[0]) * slice_size,
                    (polygon[:, 1] / bool_mask_np.shape[0]) * slice_size
                ], axis=-1)

            min_edge_distance = 0.05 * slice_size
            max_edge_distance = 0.95 * slice_size
            is_within = (
                (polygon_slice[:, 0].min() >= min_edge_distance and polygon_slice[:, 0].max() <= max_edge_distance) and
                (polygon_slice[:, 1].min() >= min_edge_distance and polygon_slice[:, 1].max() <= max_edge_distance)
            )

            if not is_within:
                score = 0.10

            shifted_polygon = shift_polygon(polygon_slice, shift_x, shift_y)

            scores.append(score)
            polygons.append(shifted_polygon)
            category_ids.append(category_id)
            category_names.append(category_name)
            is_within_slice_list.append(is_within)

        except Exception:
            continue

    return pd.DataFrame({
        'score': scores,
        'polygon': polygons,
        'category_id': category_ids,
        'category_name': category_names,
        'is_within_slice': is_within_slice_list
    })


# --- Updated get_sliced_prediction with batched inference ---
def get_sliced_predictionfast(in_raster,
                          detection_model=None,
                          confidence_threshold: float = 0.1,
                          has_mask=True,
                          output_dir=None,
                          interim_file_name=None,
                          interim_dir=None,
                          slice_size: int = None,
                          inference_size: int = None,
                          overlap_height_ratio: float = 0.2,
                          overlap_width_ratio: float = 0.2,
                          min_area_threshold: int = None,
                          downscale_pred: bool = False,
                          postprocess: bool = True,
                          postprocess_match_threshold: float = 0.5,
                          postprocess_class_agnostic: bool = False,
                          batch_size: int = 8):  # new param
    """
    Same as original but runs inference in batches (batch_size).
    """

    # convert in_raster tif file to png file
    in_raster = Path(in_raster)
    output_dir = Path(output_dir)
    out_png = in_raster.with_name(in_raster.stem + ".png")
    raster_convert.tiff_to_png(in_raster, out_png) # only work with 8bit

    # create temporary directory
    tmp_dir = (Path.home() / "tmp")
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # set model's confidence_threshold and inference size
    detection_model.image_size = inference_size
    detection_model.confidence_threshold = confidence_threshold

    slice_image_result = slice_image(
        image=out_png.as_posix(),
        output_file_name=interim_file_name,
        output_dir=interim_dir,
        slice_height=slice_size,
        slice_width=slice_size,
        overlap_height_ratio=overlap_height_ratio,
        overlap_width_ratio=overlap_width_ratio,
        out_ext=".png",
    )

    num_slices = len(slice_image_result)
    shift_amounts = slice_image_result.starting_pixels
    frames = []

    # build list of images (as numpy arrays or file paths depending on your detection_model)
    # Keep same element order as shift_amounts
    slice_images = slice_image_result.images  # original list

    # iterate in batches
    for i in tqdm(range(0, num_slices, batch_size)):
        batch_images = slice_images[i:i + batch_size]
        batch_shifts = shift_amounts[i:i + batch_size]

        # run model once for the entire batch
        # detection_model.model should accept list of images for batch inference
        prediction_results = detection_model.model(batch_images, imgsz=detection_model.image_size,
                                                   verbose=False, device=detection_model.device)

        # prediction_results should be iterable with one result per image in batch
        # process each result with the YOLOv8() function (which now expects a single result)
        for j, prediction_result in enumerate(prediction_results):
            # compute index of the slice in the global list
            global_idx = i + j
            shift = batch_shifts[j]

            df = YOLOv8fastv2(prediction_result, slice_images[global_idx], detection_model, has_mask,
                        shift, slice_size, min_area_threshold, downscale_pred)
            if df.shape[0] > 0:
                frames.append(df)

    # if no frames found, return empty gdf early (preserve original behavior)
    if len(frames) == 0:
        df_all = pd.DataFrame(columns=['score', 'polygon', 'category_id', 'category_name', 'is_within_slice'])
    else:
        df_all = pd.concat(frames, ignore_index=True)

    gdf = add_geometries(in_raster, df_all)


    # keep edge predictions (within 10% of slice size from the true footprint edge)

    # extract true footprint
    gdf_true_footprint = raster.true_footprint(in_raster, tmp_dir / "true-footprint.shp")
    in_res = raster_metadata.get_resolution(in_raster)[0]
    in_meta = raster_metadata.get_profile(in_raster)
    gpd.GeoDataFrame(geometry=gdf_true_footprint.geometry.boundary.values, crs=in_meta["crs"].to_wkt()).to_file(
        tmp_dir / "true-footprint-as-a-line.shp")
    gdf_line_buffer = shp.buffer(tmp_dir / "true-footprint-as-a-line.shp", slice_size * 0.10 * in_res,
                                 (tmp_dir / "footprint-buffer.shp"))

    gdf_boulders = gdf.copy()
    gdf_boulders["id"] = gdf_boulders.index
    gdf_intersected = gpd.overlay(gdf_boulders, gdf_line_buffer, how="intersection", keep_geom_type=True)

    gdf["is_at_edge"] = False
    gdf.loc[gdf_intersected.id.values, "is_at_edge"] = True

    # keep edge predictions close to the edge of the footprint of the raster, but otherwise remove edge predictions
    gdf = gdf.loc[np.logical_or(gdf.is_at_edge == True, gdf.is_within_slice == True)]

    # remove duplicates
    gdf = gdf.drop_duplicates(subset="geometry", ignore_index=True)
    gdf["id"] = gdf.index

    # save shapefile before post-processing (include if downscaling is done or not...)
    bbox_filename = in_raster.stem + "-predictions-ct-" + str(int(confidence_threshold * 100)).zfill(3) + "-ss-" + str(
        slice_size) + "-is-" + str(inference_size) + "-ov-" + str(int(overlap_height_ratio * 100)).zfill(3) + "-bbox.shp"
    mask_filename = bbox_filename.replace("-bbox.shp", "-mask.shp")

    if downscale_pred:
        bbox_filename = bbox_filename.replace("-bbox.shp", "-downscaled-bbox.shp")
        mask_filename = mask_filename.replace("-mask.shp", "-downscaled-mask.shp")

    out_bbox_shp = output_dir / bbox_filename
    out_mask_shp = output_dir / mask_filename
    bboxes_to_shp(gdf, out_bbox_shp)
    outlines_to_shp(gdf, out_mask_shp)


    if postprocess:
        # Non-maximum suppression (NMS)
        if postprocess_class_agnostic:
            keep = nms(boxes=np.stack(gdf.bbox.values), scores=gdf.score.values,
                       iou_threshold=postprocess_match_threshold, class_ids=None, rtree_leaf_size=32)
        else:
            keep = nms(boxes=np.stack(gdf.bbox.values), scores=gdf.score.values,
                       iou_threshold=postprocess_match_threshold, class_ids=gdf.category_id.values, rtree_leaf_size=32)

        # saving post-processed shapefiles
        gdf_nms = gdf.loc[keep]
        bboxes_to_shp(gdf_nms, out_bbox_shp.with_name(out_bbox_shp.stem + "-nms.shp"))
        outlines_to_shp(gdf_nms, out_mask_shp.with_name(out_mask_shp.stem + "-nms.shp"))
        return gdf, gdf_nms
    else:
        return gdf, None


In [ ]:
slice_sizes = [512] 
confidence_threshold = 0.10
inference_size = 1024
overlap_height_ratio = 0.20

for slice_size in slice_sizes:
        __, __ = get_sliced_predictionfast(in_raster,
                            detection_model=detection_model,
                            confidence_threshold=confidence_threshold,
                            has_mask=True,
                            output_dir=output_dir,
                            #interim_file_name=in_raster.stem,  # YOU CAN OPTIONALLY SAVE SLICES WITH SPECIFIC NAME
                            #interim_dir=interim_dir,  # YOU CAN OPTIONALLY SAVE SLICES TO THIS INTERIM DIRECTORY
                            slice_size=slice_size,
                            inference_size=inference_size,
                            overlap_height_ratio=overlap_height_ratio,
                            overlap_width_ratio=overlap_height_ratio,
                            min_area_threshold=6,
                             downscale_pred=True, # if True, the predicted mask is downscaled to the slice_size, decreasing the coputational time.  
                            postprocess= True,
                            postprocess_match_threshold=0.2, # 
                            postprocess_class_agnostic=False)

100%|██████████| 236/236 [33:00<00:00,  8.39s/it]


It takes about 30 min for a NAC image (if four different slice sizes are run).

### Crater detection

In [9]:
url_yolov8_crater_model_weights = "https://drive.google.com/uc?id=10lcw015kekhwZtDa0u7VuzgqG7NuObhZ"

In [12]:
gdown.download(url_yolov8_crater_model_weights, (model_dir / "yolov8-m-crater-detection-tmp.pt").as_posix(), quiet=True)

'C:/Users/nilscp/tmp/YOLOv8BeyondEarth/yolov8_model/yolov8-m-crater-detection-tmp.pt'

In [13]:
model_weights_crater = model_dir / "yolov8-m-crater-detection-tmp.pt"

In [14]:
crater_detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=model_weights_crater.as_posix(),
    confidence_threshold=0.1, # this parameter will be changed by the function
    device="cuda:0",  # or cpu, please run this code on a computer with a graphical card, otherwise it will take a long time! 
    image_size=1024) # this parameter will be changed by the function

In [15]:
output_dir = (work_dir / "inference_crater")
output_dir.mkdir(parents=True, exist_ok=True)

In [17]:
slice_sizes = [256] 
confidence_threshold = 0.10
inference_size = 512
overlap_height_ratio = 0.20

for slice_size in slice_sizes:
        __, __ = get_sliced_prediction(in_raster,
                            detection_model=crater_detection_model,
                            confidence_threshold=confidence_threshold,
                            has_mask=True,
                            output_dir=output_dir,
                            interim_file_name=None,  # YOU CAN OPTIONALLY SAVE SLICES WITH SPECIFIC NAME
                            interim_dir=None,  # YOU CAN OPTIONALLY SAVE SLICES TO THIS INTERIM DIRECTORY
                            slice_size=slice_size,
                            inference_size=inference_size,
                            overlap_height_ratio=0.2,
                            overlap_width_ratio=0.2,
                            min_area_threshold=6,
                            downscale_pred=True, # if True, the predicted mask is downscaled to the slice_size, decreasing the coputational time.  
                            postprocess= True,
                            postprocess_match_threshold=0.2, # 
                            postprocess_class_agnostic=False)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 7225/7225 [02:37<00:00, 46.01it/s]
